<a href="https://colab.research.google.com/github/Andresflorezdev/DAT-05_ANALISIS_y_Visualizacion_de_Datos-Esumer-Medellin/blob/main/Laboratorio_Clase_10_Estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 10 · Integración de fuentes
**ESUMER / Estud-IA · Johan Steev Muriel Mosquera**

**Objetivo de aprendizaje:** Comprender cómo llaves, tipos de unión y cardinalidad afectan la estructura de una integración y evaluar cobertura, unicidad y conservación de indicadores.

Institución ficticia. Datos sintéticos. Cada fila de matrículas representa una matrícula, no una persona única. Valor neto calculado después del descuento, no recaudo efectivo.

## 0. Conexión a Google Drive
Antes de ejecutar, crea `Curso_Analitica/Clase_10` en Mi unidad y coloca allí: `Matriculas_2026-1.csv`, `Matriculas_2026-2.csv`, `Maestro_estudiantes.csv`, `Catalogo_programas.csv`. Abre una copia del notebook en Colab. La siguiente es la primera celda de código.

In [1]:
try:
    from google.colab import drive
except ImportError:
    EN_COLAB = False
    print('Entorno local: se utiliza la carpeta del notebook.')
else:
    EN_COLAB = True
    drive.mount('/content/drive')

Mounted at /content/drive


## Cómo avanzar
Ejecuta en orden. Lee la pregunta y la regla antes de programar. Las ayudas son opcionales y los resultados esperados permiten verificar sin copiar una solución. Los controles deben ejecutarse después de completar el ejercicio.

Conserva tu notebook con tu nombre. Si no alcanzas a terminar en la sesión, continúa el mismo recorrido con estas instrucciones. No ejecutes controles de ejercicios que aún no has completado. Al reiniciar, vuelve a ejecutar desde el montaje.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
CARPETA = Path('/content/drive/MyDrive/Analisis_de_Datos_Esumer/Clase_10') if EN_COLAB else Path.cwd()
SALIDAS = CARPETA / 'Salidas'
SALIDAS.mkdir(exist_ok=True)
print('Pandas:', pd.__version__)
def leer(nombre):
    ruta = CARPETA / nombre
    if not ruta.exists():
        raise FileNotFoundError(f'Revisa la carpeta y el archivo: {ruta}')
    return pd.read_csv(ruta, encoding='utf-8-sig', dtype={'id_matricula':'string', 'id_estudiante':'string'})


Pandas: 2.2.3


## 1. Contrato de las fuentes
Las matrículas están preparadas como en la clase 8; no necesitas ejecutar otros notebooks. Maestro y catálogo son complementos sintéticos.

| Fuente | Llave esperada | Regla |
|---|---|---|
| Matrículas | id_matricula | Única en ambos periodos juntos. Un estudiante puede tener varias. |
| Estudiantes | id_estudiante | Quitar espacios exteriores. Separar llave vacía, conservar una copia exacta y separar TODAS las versiones contradictorias. No escoger una arbitrariamente. |
| Programas | programa | Catálogo único. Programa desconocido o ausente del catálogo queda sin emparejar. |

Las integraciones mantienen una fila por matrícula. No rellenes identificadores desconocidos con una llave común: Pandas puede emparejar llaves nulas. Se excluyen llaves vacías de los maestros antes de unir. Los faltantes de cobertura quedan visibles.

In [3]:
p1=leer('Matriculas_2026-1.csv')
p2=leer('Matriculas_2026-2.csv')
estudiantes=leer('Maestro_estudiantes.csv')
programas=leer('Catalogo_programas.csv')
for nombre,tabla in [('Periodo 1',p1),('Periodo 2',p2),('Estudiantes',estudiantes),('Programas',programas)]:
    print(nombre,tabla.shape)
    display(tabla.head(3))

Periodo 1 (59, 32)


,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_bruto_matricula,...,fecha_matricula_fuente,fecha_inicio_fuente,revisar_fecha,revisar_beca,valor_descuento,valor_neto_matricula,rango_edad,dias_anticipacion,tipo_anticipacion,mes_matricula
0,M0001,E001,2026-1,Analítica,Robledo,58.0,2026-01-08,2026-01-25,Activa,1200000.0,...,2026-01-08,2026-01-25,False,False,240000.0,960000.0,45–59,17.0,Anticipada,2026-01
1,M0002,E002,2026-1,Marketing,Centro,33.0,2026-01-12,2026-01-25,Cancelada,1400000.0,...,2026-01-12,2026-01-25,False,False,0.0,1400000.0,25–34,13.0,Cercana al inicio,2026-01
2,M0003,E003,2026-1,Desarrollo Web,Virtual,NaN,2026-01-07,2026-01-25,Finalizada,1800000.0,...,2026-01-07,2026-01-25,False,False,0.0,1800000.0,Sin información,18.0,Anticipada,2026-01


Periodo 2 (60, 32)


,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_bruto_matricula,...,fecha_matricula_fuente,fecha_inicio_fuente,revisar_fecha,revisar_beca,valor_descuento,valor_neto_matricula,rango_edad,dias_anticipacion,tipo_anticipacion,mes_matricula
0,M0061,E001,2026-2,Analítica,Robledo,21.0,2026-07-20,2026-07-25,Activa,1400000,...,2026-07-20,2026-07-25,False,False,0.0,1400000.0,16–24,5.0,Cercana al inicio,2026-07
1,M0062,E002,2026-2,Marketing,Centro,50.0,2026-07-07,2026-07-25,Cancelada,1400000,...,2026-07-07,2026-07-25,False,False,0.0,1400000.0,45–59,18.0,Anticipada,2026-07
2,M0063,E003,2026-2,Desarrollo Web,Virtual,56.0,2026-07-07,2026-07-25,Finalizada,1400000,...,2026-07-07,2026-07-25,False,False,420000.0,980000.0,45–59,18.0,Anticipada,2026-07


Estudiantes (62, 3)


,id_estudiante,municipio,situacion_laboral
0,E001,Medellín,No trabaja
1,E002,Bello,Trabaja
2,E003,Itagüí,Trabaja


Programas (5, 3)


,programa,area_formacion,modalidad_programa
0,Analítica,Datos,Virtual
1,Marketing,Negocios,Híbrida
2,Desarrollo Web,Tecnología,Virtual


## Ejercicio 1 · Consolidar periodos
Verifica que p1 y p2 tienen las mismas columnas y su periodo correcto. Crea `matriculas` con concatenación vertical y nuevo índice. Comprueba unicidad de id_matricula. `ignore_index` cambia el índice, no elimina duplicados.

**Resultado esperado:** 119 filas sin multiplicar ni perder matrículas.

<details><summary>Ayuda de sintaxis</summary>

`pd.concat([tabla_a, tabla_b], axis=0, ignore_index=True)`

</details>

In [4]:
# COMPLETA AQUÍ
matriculas = pd.concat([p1, p2], axis=0, ignore_index=True)

matriculas['id_matricula'].is_unique

True

In [5]:
assert len(matriculas)==len(p1)+len(p2)==119
assert matriculas['id_matricula'].notna().all() and matriculas['id_matricula'].is_unique
assert set(matriculas.columns)==set(p1.columns)
print('Universo consolidado: 119 matrículas.')

Universo consolidado: 119 matrículas.


## Ejercicio 2 · Preparar el maestro
Crea `est_trabajo` normalizando identificadores de texto. Separa `llaves_vacias`. Luego separa `copias_est` como copias exactas y elimina solo esas copias. En la tabla restante identifica llaves repetidas y separa todas sus versiones en `conflictos_est`. Guarda el resto en `est_validos`. Conserva las fuentes originales.

**Resultado esperado:** 1 llave vacía, 1 copia exacta y 2 versiones de una llave contradictoria. E003 queda pendiente y E001 se recupera al quitar espacios.

<details><summary>Ayuda de sintaxis</summary>

```python
s.astype('string').str.strip().replace('', pd.NA)
tabla.duplicated()  # copias exactas
tabla.duplicated('llave', keep=False)  # todas las versiones de una llave
```

</details>

In [6]:
# COMPLETA AQUÍ
est_trabajo = estudiantes.copy()
est_trabajo['id_estudiante'] = est_trabajo['id_estudiante'].astype('string').str.strip().replace('', pd.NA)

llaves_vacias = est_trabajo[est_trabajo['id_estudiante'].isna()]
sin_vacios = est_trabajo[est_trabajo['id_estudiante'].notna()]

copias_est = sin_vacios[sin_vacios.duplicated()]
sin_copias = sin_vacios.drop_duplicates()

conflictos_mascara = sin_copias.duplicated('id_estudiante', keep=False)
conflictos_est = sin_copias[conflictos_mascara]
est_validos = sin_copias[~conflictos_mascara]

In [7]:
assert len(llaves_vacias)==1 and len(copias_est)==1 and len(conflictos_est)==2
assert est_validos['id_estudiante'].notna().all() and est_validos['id_estudiante'].is_unique
assert len(estudiantes)==len(llaves_vacias)+len(copias_est)+len(conflictos_est)+len(est_validos)
assert 'E003' not in set(est_validos['id_estudiante'])
assert 'E001' in set(est_validos['id_estudiante'])
print('Maestro listo, conflictos conservados para revisión.')

Maestro listo, conflictos conservados para revisión.


## Ejercicio 3 · Integrar conservando matrículas
Crea `integrada_est` con unión izquierda por id_estudiante, relación muchos a uno e indicador llamado `cruce_estudiante`. Crea `sin_estudiante` con registros left_only. No clasifiques por municipio faltante: un registro podría emparejar y tener un atributo desconocido.

**Resultado esperado:** 119 filas. Hay matrículas sin emparejar por ausencia de estudiante o por conflicto separado del maestro.

<details><summary>Ayuda de sintaxis</summary>

```python
izquierda.merge(derecha, on='llave', how='left', validate='many_to_one', indicator='nombre_control')
```

</details>

In [8]:
# COMPLETA AQUÍ
integrada_est = matriculas.merge(est_validos, on='id_estudiante', how='left', validate='many_to_one', indicator='cruce_estudiante')
sin_estudiante = integrada_est[integrada_est['cruce_estudiante'] == 'left_only']

In [9]:
assert len(integrada_est)==119 and integrada_est['id_matricula'].is_unique
esperado_ids=set(matriculas['id_estudiante'])-set(est_validos['id_estudiante'])
assert set(sin_estudiante['id_estudiante'])==esperado_ids
assert sin_estudiante['cruce_estudiante'].eq('left_only').all()
assert np.isclose(integrada_est['valor_neto_matricula'].sum(),matriculas['valor_neto_matricula'].sum())
print('Universo y valor neto conservados; pendientes identificados.')

Universo y valor neto conservados; pendientes identificados.


## Ejercicio 4 · Enriquecer programas y auditar catálogo
Comprueba que programa es único y no nulo en el catálogo. Crea `integrada` con unión izquierda y `cruce_programa`. Crea `sin_programa` con left_only. Construye `auditoria_catalogo` con outer entre programas usados distintos y catálogo; identifica `catalogo_sin_uso` como right_only. El outer es diagnóstico y no reemplaza la salida principal.

**Resultado esperado:** Diseño está en catálogo sin uso. Especialización no está en catálogo; los programas desconocidos también quedan sin emparejar.

<details><summary>Ayuda de sintaxis</summary>

Usa `dropna().drop_duplicates()` para obtener programas conocidos usados. `indicator` puede tener un nombre distinto en cada cruce.

</details>

In [10]:
# COMPLETA AQUÍ
# Comprobar que programa es único y no nulo en el catálogo
assert programas['programa'].is_unique
assert programas['programa'].notna().all()

# Crear integrada con unión izquierda y cruce_programa
integrada = integrada_est.merge(programas, on='programa', how='left', indicator='cruce_programa')

# Crear sin_programa con left_only
sin_programa = integrada[integrada['cruce_programa'] == 'left_only']

# Construir auditoria_catalogo con outer entre programas usados distintos y catálogo
programas_usados = matriculas[['programa']].dropna().drop_duplicates()
auditoria_catalogo = programas_usados.merge(programas, on='programa', how='outer', indicator='cruce_auditoria')

# Identificar catalogo_sin_uso como right_only
catalogo_sin_uso = auditoria_catalogo[auditoria_catalogo['cruce_auditoria'] == 'right_only']

In [11]:
assert len(integrada)==119 and integrada['id_matricula'].is_unique
assert set(catalogo_sin_uso['programa'])=={'Diseño'}
assert sin_programa['programa'].isna().any()
assert 'Especialización' in set(sin_programa['programa'].dropna())
assert set(integrada['id_matricula'])==set(matriculas['id_matricula'])
print('Catálogo y cobertura revisados sin cambiar el universo.')

Catálogo y cobertura revisados sin cambiar el universo.


## Ejercicio 5 · ¿Concat horizontal integra personas?
Ejecuta la celda de demostración inferior después de resolver: crea `horizontal_segura` concatenando A y B con id_estudiante como índice; devuelve la llave a columna. Escribe por qué resetear índices en ambas tablas antes de concatenar no asegura correspondencia.

**Resultado esperado:** E001 debe quedar con Correo y E002 con SMS. Igual cantidad de filas no demuestra alineación.

<details><summary>Ayuda de sintaxis</summary>

`pd.concat([A.set_index("id_estudiante"), B.set_index("id_estudiante")], axis=1).reset_index()`

</details>

In [14]:
# COMPLETA AQUÍ
A = pd.DataFrame({'id_estudiante': ['E001', 'E002'], 'canal': ['Web', 'Sede']})
B = pd.DataFrame({'id_estudiante': ['E002', 'E001'], 'contacto': ['SMS', 'Correo']})

horizontal_segura = pd.concat([A.set_index('id_estudiante'), B.set_index('id_estudiante')], axis=1).reset_index()

# Explicación:
# Resetear los índices numéricos antes de concatenar horizontalmente (axis=1) hace que la unión se realice estrictamente por la posición física de las filas.
# Si el orden de los estudiantes varía entre ambas tablas (por ejemplo, E001 está primero en A pero de segundo en B), se emparejarán datos de personas distintas
# solo porque comparten la misma posición de fila, lo cual altera y corrompe completamente la integridad de los datos.

In [15]:
A=pd.DataFrame({'id_estudiante':['E001','E002'],'canal':['Web','Sede']})
B=pd.DataFrame({'id_estudiante':['E002','E001'],'contacto':['SMS','Correo']})
print('Resultado por posición/índice numérico:')
display(pd.concat([A,B],axis=1))
assert horizontal_segura.set_index('id_estudiante').loc['E001','contacto']=='Correo'
assert horizontal_segura.set_index('id_estudiante').loc['E002','contacto']=='SMS'
print('Alineación por índice de persona verificada.')

Resultado por posición/índice numérico:


,id_estudiante,canal,id_estudiante,contacto
0,E001,Web,E002,SMS
1,E002,Sede,E001,Correo


Alineación por índice de persona verificada.


## Ejercicio 6 · Reporte de consistencia
Crea `control` por periodo comparando antes y después: filas, matrículas únicas, cantidad de netos conocidos y suma de netos conocidos. Crea `cobertura` con una fila por cruce y columnas `cruce`, `emparejadas`, `no_emparejadas` y `cobertura_pct`. Compara también el neto por id_matricula para que una suma igual no oculte cambios individuales.

**Resultado esperado:** Las métricas antes/después coinciden; cobertura inferior al 100 % se informa, no se oculta.

<details><summary>Ayuda de sintaxis</summary>

Reutiliza agregaciones y concat horizontal cuando el índice es periodo. Para comparar por matrícula, usa `.set_index("id_matricula").sort_index()`.

</details>

In [16]:
# COMPLETA AQUÍ
gp_antes = matriculas.groupby('periodo')
gp_despues = integrada.groupby('periodo')

control = pd.DataFrame({
    'filas_antes': gp_antes.size(),
    'filas_despues': gp_despues.size(),
    'unicas_antes': gp_antes['id_matricula'].nunique(),
    'unicas_despues': gp_despues['id_matricula'].nunique(),
    'netos_antes': gp_antes['valor_neto_matricula'].count(),
    'netos_despues': gp_despues['valor_neto_matricula'].count(),
    'neto_antes': gp_antes['valor_neto_matricula'].sum(),
    'neto_despues': gp_despues['valor_neto_matricula'].sum()
})

cobertura = pd.DataFrame([
    {
        'cruce': 'cruce_estudiante',
        'emparejadas': integrada['cruce_estudiante'].eq('both').sum(),
        'no_emparejadas': integrada['cruce_estudiante'].ne('both').sum(),
        'cobertura_pct': (integrada['cruce_estudiante'].eq('both').sum() / 119) * 100
    },
    {
        'cruce': 'cruce_programa',
        'emparejadas': integrada['cruce_programa'].eq('both').sum(),
        'no_emparejadas': integrada['cruce_programa'].ne('both').sum(),
        'cobertura_pct': (integrada['cruce_programa'].eq('both').sum() / 119) * 100
    }
])

In [17]:
for metrica in ['filas','unicas','netos','neto']:
    assert np.allclose(control[metrica+'_antes'],control[metrica+'_despues'])
pd.testing.assert_series_equal(matriculas.set_index('id_matricula')['valor_neto_matricula'].sort_index(),integrada.set_index('id_matricula')['valor_neto_matricula'].sort_index())
assert cobertura['emparejadas'].add(cobertura['no_emparejadas']).eq(119).all()
for _,r in cobertura.iterrows():
    assert r['emparejadas']==integrada[r['cruce']].eq('both').sum()
    assert np.isclose(r['cobertura_pct'],r['emparejadas']/119*100)
print('Controles por periodo, por matrícula y cobertura superados.')

Controles por periodo, por matrícula y cobertura superados.


## 8. Interpretación y entrega
**¿Por qué elegiste left para la salida principal?** …  
**¿Qué ocurriría con inner?** …  
**¿Qué pendientes necesitan consultar al responsable de la fuente?** …  
**¿La base puede usarse para indicadores por municipio? ¿Qué limitación debe acompañarlos?** …  
**¿Por qué conservar filas y suma no basta para probar que todas las correspondencias son correctas?** …

Registra el universo, las reglas de normalización, la cardinalidad y las coberturas. El porcentaje de cobertura usa matrículas, no estudiantes distintos.

In [18]:
for nombre,tabla in [('Matriculas_integradas',integrada),('Sin_estudiante',sin_estudiante),('Sin_programa',sin_programa),('Conflictos_estudiantes',conflictos_est),('Llaves_vacias',llaves_vacias),('Copias_estudiantes',copias_est),('Catalogo_sin_uso',catalogo_sin_uso),('Cobertura_integracion',cobertura)]:
    tabla.to_csv(SALIDAS/(nombre+'.csv'),index=False,encoding='utf-8-sig')
control.to_csv(SALIDAS/'Control_integracion.csv',encoding='utf-8-sig')
print('Resultados y pendientes guardados en:',SALIDAS)

Resultados y pendientes guardados en: /content/drive/MyDrive/Analisis_de_Datos_Esumer/Clase_10/Salidas


## Si algo falla
| Mensaje | Qué revisar |
|---|---|
| FileNotFoundError | Drive montado, carpeta y nombre exactos. |
| NameError | Ejecución de celdas anteriores y nombre de variable. |
| KeyError | Nombres de columnas con `.columns`. |
| MergeError | Llaves duplicadas y relación esperada. No elimines `validate` para ocultarlo. |
| AssertionError | Lee el mensaje y compara tus resultados con la regla. |
| Totales inflados | Duplicados en la tabla derecha o llaves insuficientes. |

## Referencias oficiales
- [Agrupaciones](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Integración y concatenación](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Google Drive en Colab](https://colab.research.google.com/notebooks/io.ipynb)
